In [4]:
import mysql.connector
import csv
import sys
from datetime import datetime

# Параметры подключения к MySQL
DB_PARAMS = {
    'database': 'forum_db',
    'user': 'user',
    'password': 'password',
    'host': 'localhost',
    'port': 5432
}

# Подключение к базе
def connect_db():
    return mysql.connector.connect(**DB_PARAMS)

# Агрегация
def aggregate_data(start_date, end_date):
    conn = connect_db()
    cur = conn.cursor()

    query = """
    SELECT 
        DATE(timestamp) AS day,
        COUNT(DISTINCT CASE WHEN action_id = 2 THEN user_id END) AS new_accounts,
        COUNT(CASE WHEN action_id = 8 THEN 1 END) AS total_messages,
        COUNT(CASE WHEN action_id = 8 AND user_id IS NULL THEN 1 END) AS anonymous_messages,
        COUNT(CASE WHEN action_id = 5 THEN 1 END) AS new_themes
    FROM logs
    WHERE timestamp BETWEEN %s AND %s
    GROUP BY day
    ORDER BY day;
    """

    print(f"Агрегируем с {start_date} по {end_date}")
    cur.execute(query, (start_date, end_date))
    rows = cur.fetchall()

    results = []
    prev_themes_total = None
    total_themes = 0

    for row in rows:
        day, new_accounts, total_messages, anonymous_messages, new_themes = row

        total_themes += new_themes
        anonymous_percentage = round((anonymous_messages / total_messages * 100), 2) if total_messages > 0 else 0

        if prev_themes_total is not None:
            theme_growth = round(((total_themes - prev_themes_total) / prev_themes_total * 100), 2) if prev_themes_total > 0 else 0
        else:
            theme_growth = 0

        results.append([str(day), new_accounts, anonymous_percentage, total_messages, theme_growth])
        prev_themes_total = total_themes

    conn.close()
    return results

# Запись в CSV
def write_to_csv(data, filename='aggregated_logs_mysql.csv'):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Day', 'New Accounts', 'Anonymous Messages (%)', 'Total Messages', 'Theme Growth (%)'])
        writer.writerows(data)

# Основная функция
def main():
    if len(sys.argv) < 3:
        print("Usage: python generation_mysql.py YYYY-MM-DD YYYY-MM-DD")
        sys.exit(1)

    start_date = sys.argv[1]
    end_date = sys.argv[2]

    try:
        datetime.strptime(start_date, '%Y-%m-%d')
        datetime.strptime(end_date, '%Y-%m-%d')
    except ValueError:
        print("Invalid date format. Use YYYY-MM-DD.")
        sys.exit(1)

    aggregated_data = aggregate_data(start_date, end_date)
    write_to_csv(aggregated_data)
    print(" Aggregation complete. Saved to 'aggregated_logs_mysql.csv'.")

if __name__ == '__main__':
    main()


OperationalError: connection to server at "localhost" (::1), port 5432 failed: FATAL:  database "forum_db" does not exist
